# Interfacing with the Duckiedrone IMU from ROS 2

Notebooks 1 to 3 covered what an IMU measures, how to describe the flight controller's
  orientation estimate, and the chain that carries those values to a ROS 2 topic.

This notebook adds the node at the end of that chain. It subscribes to the topic and prints the
  readings once a second, so that the values can be watched while the Duckiedrone is tipped.

Most of the processing happens before the data reaches the node, so the node itself stays
  short. To recap: PX4 applies the configured sensor calibration, rotates the readings into its
  body frame, and uses them to estimate orientation. MAVROS2 converts the resulting
  measurements and attitude estimate to ROS conventions before publishing them.

What remains is to ask ROS 2 for the messages and to unpack the data in them.

## Getting to a terminal that can see the Duckiedrone

The topics used below exist on the Duckiedrone itself, on the right of the vertical line in
  Notebook 3. Four steps lead to them.

A physical and a virtual Duckiedrone run the same containers, so every command from Step 2
  onward is identical on the two. Only the way the first shell is opened, in Step 1, differs.

### Step 0. Reach the Duckiedrone from the base station

Open a terminal on the base station and check that `ping <robot>.local` succeeds, where
  `<robot>` is the Duckiedrone's hostname. If it does not, nothing below will work either.
  Consult the general
  [network troubleshooting tips](https://docs.duckietown.com/ente/duckietown-manual/06-troubleshooting/duckiebot-network-faqs.html)
  in the Duckietown manual, and the
  [instructions on how to manage the Duckiedrone network(s)](https://docs.duckietown.com/ente/opmanual-dd24/40-duckiedrone-handling/70-network-configuration.html).

### Step 1. Open a shell on the Duckiedrone

On a physical Duckiedrone, connect with SSH:

```bash
ssh duckie@<robot>.local
```

`<robot>` is your Duckiedrone's hostname, and the password is the one set when its SD card was
  flashed. If none was set, the default is `quackquack`.

SSH is not fully supported for a virtual Duckiedrone. From the computer running the Duckietown
  Shell, open its shell with:

```bash
dts duckiebot virtual connect <robot>
```

This opens a root shell on the virtual Duckiedrone. `exit` returns you to the computer running
  the Duckietown Shell. Before expecting simulated sensor data, start the Duckiematrix and attach
  the Duckiedrone to an entity.

### Step 2. Find the container that publishes the IMU

The Duckiedrone does not run its software as ordinary programs. Each job runs in its own Docker
  container, with its own filesystem and its own installed libraries, so that changing one
  piece of software cannot break another. List them:

```bash
docker ps --format '{{.Names}}'
```

The names describe jobs:

```
ros2-mavros
ros2-camera
driver-camera
driver-tof-bottom
dashboard
...
```

`ros2-mavros` is the one that holds the link to the flight controller open and publishes
  `/mavros/imu/data`, which makes it Step 3 of Notebook 3. There is no IMU container in that
  list, because there is no IMU driver on the Duckiedrone: the IMU is wired to the flight
  controller, not to the Duckiedrone's computer.

### Step 3. Open a shell inside it

```bash
docker exec -it ros2-mavros bash
```

`exec` runs a command inside a container that is already running, and `-it` gives an
  interactive terminal rather than a single command. The container prints a short banner while
  it sets the shell up, the prompt changes, and `exit` returns to the host. A warning about a
  missing colcon setup file is normal here and does not affect the commands below.

ROS 2 is already configured inside the container, including the Duckiedrone's `ROS_DOMAIN_ID`,
  so `ros2 topic list` works there straight away. The ROS 2 containers share the Duckiedrone's
  host network, but a shell opened elsewhere needs ROS 2 installed, sourced, and set to the
  same domain before it can discover these topics. Without that setup it may list nothing while
  the Duckiedrone runs perfectly beside it.

A virtual Duckiedrone has the same containers and topics once you connect with `dts duckiebot
  virtual connect <robot>`. The only difference is that PX4 runs in software rather than on a
  board, as SITL, with the attached Duckiematrix entity feeding it sensor readings.

## Look at the data first

The IMU is publishing whether or not anything is listening. Three commands describe the topic
  before a line of code is written.

**What exists:**

```bash
ros2 topic list | grep imu
```

This should list `/mavros/imu/data` and `/mavros/imu/data_raw`, among a few others. If the
  names come back with a `/<robot>` prefix in front, the Duckiedrone publishes under a
  namespace and every command below needs that prefix too.

**How often it arrives:**

```bash
ros2 topic hz /mavros/imu/data
```

The MAVLink stream is configured for 50 Hz, and a physical Duckiedrone measures about that
  rate. Timing on a virtual Duckiedrone can differ, so measure it with this command rather than
  assuming the configured value.

**What is inside one message:**

```bash
ros2 topic echo --once /mavros/imu/data
```

In what `echo` prints, `orientation` is a quaternion, `angular_velocity` is in rad/s, and
  `linear_acceleration` is in m/s^2. Each group has a covariance array that describes
  uncertainty when the publisher provides it. In a `sensor_msgs/Imu` message, an all-zero
  covariance means the covariance is unknown; for `orientation_covariance`, a first entry of
  `-1` means no orientation estimate is available.

Run the same `echo` on `/mavros/imu/data_raw` and compare. Its orientation is typically zeros
  with `orientation_covariance[0]` set to `-1`, because that topic carries the accelerometer
  and gyroscope without the fused attitude, so the two topics can be compared field by field.

## Activity 1: write the subscriber

Open `packages/solution/solution/imu_subscriber.py`. Most of the file is already written: it
  creates the node, works out the topic name from the `robot_namespace` parameter, sets
  `self.latest_reading` to `None`, and starts a timer that calls `cb_log` once a second to
  print.

Two pieces are missing. Each is marked in the file with a `TO IMPLEMENT` block listing the
  names and expressions to use.

### In `__init__`, ask ROS 2 for the messages

A subscription is a standing request. It states which topic the node wants and which method
  each message should be handed to, and from then on the node never asks for a message: ROS 2
  calls the method as messages arrive. That is the arrangement from Step 4 of Notebook 3.

Two of the four arguments are worth pausing on.

The callback is passed as `self.cb_imu`, with no parentheses. Parentheses would call the method
  once, immediately, and hand its return value to the subscription. What the subscription needs
  is the method itself, to call later, many times.

The last argument is a Quality of Service profile, which describes how a topic is delivered:
  whether delivery is retried, and how many messages are buffered. A subscriber should declare
  the same profile as the publisher. MAVROS2 publishes the IMU topics as best-effort sensor
  data with a queue depth of 5, which is what `qos_profile_sensor_data` describes. Read the
  publisher's side rather than taking that on trust:

```bash
ros2 topic info -v /mavros/imu/data
```

### In `cb_imu`, unpack one message

The callback receives one `sensor_msgs/Imu` and stores its numbers in `self.latest_reading`,
  where `cb_log` finds them a second later.

Three groups of numbers come out of that message, and they do not all have the same standing.
  `linear_acceleration` and `angular_velocity` are calibrated sensor measurements.
  `orientation` is the flight controller's fused attitude estimate, computed from those
  measurements together with everything the estimator saw before them. Notebook 3 followed both
  down the same chain. They arrive in the same message, but only two of the three are
  measurements.

The orientation arrives as a quaternion, which is what MAVROS2 publishes. Quaternions have no
  singularity, but they are unreadable at a glance, so the callback converts to roll, pitch and
  yaw using the helper from `imu_math.py`.

The callback does not print. Messages arrive at about 50 Hz while a person reads about one line
  per second, so printing here would produce fifty lines a second. More importantly, ROS 2
  delivers each message by calling this method, so slow work inside it delays every message
  behind it. Storing here and printing on the timer keeps the two rates independent.

With both pieces written, the node is ready to run.

## Activity 2: run the node on the Duckiedrone

This code has to end up on the Duckiedrone in a container, alongside the ones listed with
  `docker ps` earlier. Rather than copying files by hand, `dts` synchronizes the repository,
  builds an image on the Duckiedrone, and runs it there. The container it runs is the
  Duckietown **workbench**, where the node lives.

It takes two commands, both from the root folder of this learning experience (LX). First build
  the image:

```bash
dts code build -R <robot>
```

This syncs the repository to the Duckiedrone and builds it there, so the first run takes a few
  minutes. Then start it:

```bash
dts code workbench -R <robot>
```

Once the container starts, its launcher sources ROS 2, then the workspace containing the
  packages, and runs the launch file. The node's output appears in the terminal once a second:

```
[imu_listener_node-1] [INFO] [imu_listener_node]: accel=[-0.07 -0.11 +9.75] m/s^2 | gyro=[+0.00 +0.00 -0.00] rad/s | rpy=[  -0.6   +0.4  +89.9] deg
```

The `[imu_listener_node-1]` prefix is `ros2 launch` labelling the output by process. `Ctrl-C`
  stops the node and cleans the container up. After an edit to the code, both commands are run
  again to deploy the updated version.

### Running the launch file by hand

The launcher runs `ros2 launch imu_listener imu_listener_node.launch.py` with no arguments. A
  Duckiedrone that publishes under a namespace needs that namespace passed as an argument,
  which means running the launch file by hand inside the workbench.

`--shell` attaches to a workbench that is already running rather than starting one, so this
  takes two terminals. Leave `dts code workbench -R <robot>` running in the first, and in a
  second:

```bash
dts code workbench -R <robot> --shell
```

An answer of `No such container` means there is no workbench running for it to attach to. Then,
  inside that shell:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
source ${COLCON_WS}/install/setup.bash --extend

ros2 launch imu_listener imu_listener_node.launch.py robot_namespace:=<robot>
```

The two `source` lines are what the launcher does in the normal case. The first puts the `ros2`
  command on the path, and the second adds the workspace holding `imu_listener` so that ROS 2
  can find the package.

The node prints a `Subscribing to ...` line at startup. It shows the topic the namespace
  argument produced, which is the quickest way to distinguish a namespace mistake from an
  upstream Duckiedrone problem.

### When nothing arrives

If the node prints `Waiting for IMU data...` and never stops, it is running and nothing is
  reaching it. Work back along the chain from Notebook 3:

- `ros2 node list` should show a MAVROS2 node: `/mavros` at root or `/<robot>/mavros` when the
  Duckiedrone uses that namespace. If it does not, MAVROS2 is not running and there is nothing
  to subscribe to.

- The node logs the topic it subscribed to when it starts. Compare that against
  `ros2 topic list`. A namespace mismatch is a common cause.

- `ros2 topic hz /mavros/imu/data` should report about 50 Hz on a physical Duckiedrone. Use
  `/<robot>/mavros/imu/data` when the node logs that namespaced topic. Measure the rate
  directly on a virtual Duckiedrone, because its timing can differ. Silence there means the
  problem is upstream of the node, on the link to the flight controller.

## Activity 3: tilt the Duckiedrone and watch the angles

Leave the listener running and tilt the Duckiedrone in pitch and roll, using the Duckiematrix
  controls on a virtual Duckiedrone or by hand on a physical one. Four things to check in the
  listener output:

- Roll follows the applied tilt and stays between -180 and +180 deg.

- Pitch follows too, but stays between -90 and +90 deg. That half-range is the ZYX Euler
  convention from Notebook 2, not a bug in the code.

- `accel[2]` sits near +9.81 m/s^2 when the Duckiedrone is level and still, and falls as the
  Duckiedrone tilts, because gravity is shared out across the three axes.

- Yaw has no absolute reference. Gravity can correct roll and pitch, but nothing outside the
  Duckiedrone points north, so yaw starts from an arbitrary heading and can drift.

Then set the Duckiedrone down, leave it still for a minute, and watch yaw. Note whether it
  creeps rather than jumps. Slow drift is expected when a bias in the measured angular velocity
  is integrated, but the amount depends on the estimator and the sensor. A virtual Duckiedrone's
  simulated gyroscope has no such bias, so yaw there may not drift at all.

## Going further

Each of these is optional, and each one picks up where an earlier notebook stopped:

- Subscribe to `/mavros/imu/data_raw` alongside `/mavros/imu/data` and print both. Compare the
  acceleration, angular-velocity, orientation, and covariance fields. Which values differ, and
  which topic contains the flight controller's fused orientation?

- Record the IMU while tilting the Duckiedrone, then plot roll, pitch, and yaw.
  `ros2 bag record /mavros/imu/data` writes every message on the topic to a file that can be
  replayed or exported afterwards. The recording plotted in Notebook 1 was captured this way
  and then converted to the CSV that ships with this LX.